In [0]:
# COMMAND ----------
from pyspark.sql import functions as F
from datetime import datetime

# Lecture Bronze
df = spark.table("pharma_catalog.bronze.bronze_sales_orders")
display(df)

In [0]:
# COMMAND ----------
# Déduplication
df_dedup = df.dropDuplicates(["sales_order_id"])
print(f"Avant: {df.count()} | Après dédup: {df_dedup.count()}")

# Normalisation customer_type
df_norm = df_dedup.withColumn("customer_type", F.initcap(F.col("customer_type")))

# Contrôles qualité
df_quality = df_norm.withColumn("rule_failed",
    F.when(F.col("sales_order_id").isNull(), F.lit("null_sales_order_id"))
     .when(F.col("order_date").isNull(), F.lit("null_order_date"))
     .when(F.col("product_id").isNull(), F.lit("null_product_id"))
     .when(F.col("warehouse_id").isNull(), F.lit("null_warehouse_id"))
     .when(F.col("customer_type").isNull(), F.lit("null_customer_type"))
     .when(F.col("quantity_ordered") <= 0, F.lit("quantite_invalide"))
     .when(F.col("quantity_ordered") > 10000, F.lit("quantite_suspecte"))
     .when(F.col("quantity_delivered") < 0, F.lit("livraison_negative"))
     .when(F.col("quantity_delivered") > F.col("quantity_ordered"), F.lit("livraison_superieure_commande"))
     .when(F.col("sales_amount") < 0, F.lit("montant_negatif"))
     .otherwise(F.lit(None))
)

passed = df_quality.filter(F.col("rule_failed").isNull()).drop("rule_failed")
failed = df_quality.filter(F.col("rule_failed").isNotNull())

print(f"✅ Lignes OK : {passed.count()}")
print(f"❌ Lignes rejetées : {failed.count()}")
display(failed.select("sales_order_id", "quantity_ordered", "quantity_delivered", "sales_amount", "rule_failed"))

In [0]:
# COMMAND ----------
# Voir les lignes rejetées
display(failed)

In [0]:
# COMMAND ----------
if failed.count() > 0:
    df_quarantine = failed.withColumn("source_table", F.lit("bronze_sales_orders")) \
                           .withColumn("processing_date", F.lit(datetime.now().strftime("%Y-%m-%d")))
    df_quarantine.write.mode("append") \
                       .option("mergeSchema", "true") \
                       .saveAsTable("pharma_catalog.silver_quarantine.quarantine")
    print(f" {failed.count()} ligne(s) envoyée(s) en quarantine")



In [0]:
# COMMAND ----------
# Bloc 4 — Silver Clean
passed.write.mode("overwrite").saveAsTable("pharma_catalog.silver.silver_sales_orders")
print(f" {passed.count()} ligne(s) chargée(s) dans silver_sales_orders")

In [0]:
# COMMAND ----------
display(spark.table("pharma_catalog.silver.silver_sales_orders"))

# COMMAND ----------
display(spark.table("pharma_catalog.silver_quarantine.quarantine"))